## Query Construction

The next logical step in the overall RAG pipeline is query construction. **Query Construction** refers to the process of converting the text query from user to the _appropriate format_ of the datasource which will provide the context. There could be several types of data-sources that can provide context for the user's query - for example, the source could be an RDBMS or a GraphDB or a vector DB where we embed contents of various types of _unstructured files_ (text filed, markdown files, Word documents, PDFs, Excel spreadsheets, Powerpoints etc.)

The user-query, which is usually natural language text, must be _constructed_ to a format that the data-source will understand - for example, RDBMS will require _text-to-SQL_ conversion, GraphDB will require _text-to-Cypher_ conversion and vector DBs will need a _text-to-embedding_ conversion.

In this workbook we'll illustrate various techniques for _query construction_, such as text-to-SQL, text-to-cipher and (one that we are most familiar with) text-to-embeddings.

<center>
   <img src="../images/rag_query_construction.png" width="640" height=
</center>



### Text-to-SQL
In this section of the notebook we illustrate a mechanism to convert plain text to SQL using the LangChain framework. We have developed this on a sufficiently complext database schema - the `sakila` schema, which is a DVD rental application. This is compatible with `MySQL` or `PostgreSQL` or `SQLite`. We'll use `SQLite` version to keep things simple.

This is the schema for your quick reference:
<center>
   <img src="../db/sakila.png" width="640" height="480"/>
</center>

We'll need to install the following packages in our local Python environment (e.g. show using `uv`)
```bash
$> source .venv/bin/activate
$> uv add python_dotenv rich sqlparse tabulate langchain_core langchain_community
```

In [13]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from sqlalchemy import text
from tabulate import tabulate

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, Runnable
from langchain_core.output_parsers import StrOutputParser
from langchain_community.utilities import SQLDatabase
from langchain.prompts import ChatPromptTemplate

In [14]:
# to suppress specific warnings I get from SQLAlchemy
import warnings
from sqlalchemy.exc import SAWarning

# Suppress the specific SAWarning related to unresolvable cycles
warnings.filterwarnings(
    "ignore",
    category=SAWarning,
    message="Cannot correctly sort tables; there are unresolvable cycles between tables .*",
)

In [15]:
# load all API keys from .env file
load_dotenv(override=True)
# for colorful text
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [16]:
# create our LLM - we'll be using Claude Sonnet, but you can use LLM of your choice
# llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
llm = init_chat_model("claude-3-7-sonnet-20250219", model_provider="anthropic")

In [17]:
# load the SQLite database
db_path = Path(os.getcwd()) / ".." / "db" / "sakila_master.db"
db = None

if not db_path.exists():
    raise FileNotFoundError(f"Database file not found at {db_path}")
else:
    print(f"Loading database from {db_path}")
    db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

Loading database from C:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\Advanced RAG\..\db\sakila_master.db


In [18]:
def get_schema():
    """Get the database schema as a string."""
    schema = db.get_table_info()
    return schema


def run_query(sql_query: str):
    """Run a SQL query against the database and return the results.
    @params:
        -sql_query - the SQL query to run ("select....")
    @returns
        - (column_names, rows) - tuple with column names & rows from select
    """
    # results = db.run(sql_query)
    # return results

    try:
        # Use SQLDatabase.run to execute the query
        # NOTE: db.run returns the results as a string.
        # We must use the underlying connection for structured results.

        # Get the underlying connection
        with db._engine.connect() as connection:
            # Execute the query
            # result = connection.execute(sqlparse.parse(sql_query)[0])
            result = connection.execute(text(sql_query))

            # Fetch results
            column_names = list(result.keys())
            rows = result.fetchall()

            return (column_names, rows)
    except Exception as e:
        # Important to catch errors during execution
        # return (["Error"], [f"SQL Execution Failed: {e}"])
        return f"SQL_EXECUTION_ERROR: {e}"


def run_query_to_markdown(sql_query: str):
    # fetch results
    results = run_query(sql_query)
    # convert to markdown table using tabulate

    # 1. Check if the input is the string error flag from the Executor
    if isinstance(results, str) and results.startswith("SQL_EXECUTION_ERROR:"):
        error_message = results.replace("SQL_EXECUTION_ERROR: ", "")
        return f"**Error Executing Query:**\n```\n{error_message}\n```"

    # 2. We expect a tuple with 2 elements
    if not isinstance(results, tuple) or len(results) != 2:
        return "Error: Invalid results format passed to MarkdownTableParser."

    column_names, rows = results

    if not rows:
        return "**No results found for the query.**"

    # use tabulate to format into markdown table
    table = tabulate(
        rows,
        headers=column_names,
        tablefmt="pipe",
    )
    return table


class SQLQueryExecutor(Runnable):
    """Runnable to execute the SQL query against the database."""

    def __init__(self, db: SQLDatabase):
        self.db = db

    def invoke(self, sql_query: str, config=None):
        """
        Executes the SQL query string against the database.
        Returns a tuple: (column_names, results)
        """
        try:
            # Use SQLDatabase.run to execute the query
            # NOTE: db.run returns the results as a string.
            # We must use the underlying connection for structured results.

            # Get the underlying connection
            with self.db._engine.connect() as connection:
                # Execute the query
                # result = connection.execute(sqlparse.parse(sql_query)[0])
                result = connection.execute(text(sql_query))

                # Fetch results
                column_names = list(result.keys())
                rows = result.fetchall()

                return (column_names, rows)
        except Exception as e:
            # Important to catch errors during execution
            # return (["Error"], [f"SQL Execution Failed: {e}"])
            return f"SQL_EXECUTION_ERROR: {e}"

In [19]:
from langchain.schema import BaseOutputParser
import sqlparse
import re


class SQLParserAndFormatter(BaseOutputParser[str]):
    """Parse generated SQL and neatly format it"""

    def parse(self, text: str) -> str:
        # Use regex to replace all sequences of whitespace (including newlines)
        # with a single space, and then strip leading/trailing spaces.
        pre_formatted_sql = re.sub(r"\s+", " ", text).strip()
        # format it using sqlparse
        formatted_sql = sqlparse.format(
            pre_formatted_sql,
            reindent=True,
            keyword_case="upper",
            indent_width=4,
        )
        return formatted_sql.strip()


class MarkdownTableParser(BaseOutputParser[str]):
    """Output parser that converts (columns, rows) tuple OR error string into a Markdown table."""

    def parse(self, data) -> str:
        # 1. Check if the input is the string error flag from the Executor
        if isinstance(data, str) and data.startswith("SQL_EXECUTION_ERROR:"):
            error_message = data.replace("SQL_EXECUTION_ERROR: ", "")
            return f"**Error Executing Query:**\n```\n{error_message}\n```"

        # 2. Proceed with successful tuple parsing (original logic)
        if not isinstance(data, tuple) or len(data) != 2:
            return "Error: Invalid data format passed to MarkdownTableParser."

        column_names, rows = data

        if not rows:
            return "**No results found for the query.**"

        # ... (tabulate code unchanged) ...
        table = tabulate(
            rows,
            headers=column_names,
            tablefmt="pipe",
        )
        return table

In [ ]:
template = """Based on the table schema below, write a SQL query that would
nswer the user's question:

{schema}

Question: {question}
SQL Query:"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a SQL query."
            "Do not return anything else apart from the SQL query, no prefix or suffix quotes, "
            "no additional text apart from SQL query. Keep all SQL keywords in uppercase",
        ),
        ("human", template),
    ]
)


def text_to_sql(llm, prompt=prompt) -> Runnable:
    """converts user-query in natural languaue (text) to SQL based on database schema"""

    return (
        RunnablePassthrough.assign(schema=lambda x: get_schema())
        | prompt
        | llm
        | StrOutputParser()
        | SQLParserAndFormatter()
    )


def execute_and_format_query(sql_query: str) -> str:
    """
    Combines the SQL execution and the Markdown table formatting.
    This bypasses the tuple-output issue by keeping the logic inside a single step.
    """

    # Instantiate the executor and run it
    executor = SQLQueryExecutor(db)
    results_tuple_or_error_string = executor.invoke(sql_query)

    # Instantiate the formatter and parse the executor's result
    formatter = MarkdownTableParser()
    final_output = formatter.parse(results_tuple_or_error_string)

    return final_output


def text_to_sql_and_run(llm, prompt=prompt) -> Runnable:
    """converts user-query in natural languaue (text) to SQL based on database schema"""

    text_to_sql_chain = (
        RunnablePassthrough.assign(schema=lambda x: get_schema())
        | prompt
        | llm
        | StrOutputParser()
        | SQLParserAndFormatter()
        # The output here is the formatted SQL string
    )

    # --- CORRECTED CHAIN ---
    # Replace the two separate custom steps with the single RunnableLambda
    execution_step = RunnableLambda(execute_and_format_query)
    execution_chain = text_to_sql_chain | execution_step

    return execution_chain

Some queries you can try (in natural language):
1. List the names of all actors (last name & first name) that acted in film AMADEUS HOLY
2. In which stores is the film AMADEUS HOLY available? List the store ID, address and manager name
3. List names and date rented of all customers who have rented the movie AMADEUS HOLY along with the store they rented from (store id and address)
4. Show me by store ID and address how many times the movie AMADEUS HOLY has been rented out?

In [21]:
# user_query = "List names and date rented of all customers who have rented the movie AMADEUS HOLY along with the store they rented from (store id and address)"
user_query = "List all actors (last name & first name) whose last name starts with 'D' and the count of movies they have acted in, order the result by movie count from largest to smallest"

generated_sql = text_to_sql(llm, prompt).invoke({"question": user_query})
# NOTE: we need not really have LLM generate the results, it can be a direct
# db call, using usual tools - will save LLM call & cost to customer and could be faster!
# response = text_to_sql_and_run(llm, prompt).invoke({"question": user_query})
response = run_query_to_markdown(generated_sql)
console.print(f"\n[bold green]SQL:[/bold green]\n{generated_sql}")
console.print(f"\n[bold blue]Result:[/bold blue]")
console.print(Markdown(response))


SQL:
SELECT a.last_name,
       a.first_name,
       COUNT(fa.film_id) AS movie_count
FROM actor a
JOIN film_actor fa ON a.actor_id = fa.actor_id
WHERE a.last_name LIKE 'D%'
GROUP BY a.actor_id,
         a.last_name,
         a.first_name
ORDER BY movie_count DESC

Result:

                                        
  last_name   first_name   movie_count  
 ────────────────────────────────────── 
  DEGENERES   GINA                  42  
  DAMON       SCARLETT              36  
  DUNST       GROUCHO               35  
  DAVIS       SUSAN                 33  
  DENCH       JULIANNE              32  
  DEAN        RIVER                 31  
  DUKAKIS     ROCK                  30  
  DUKAKIS     BURT                  29  
  DEGENERES   JODIE                 29  
  DREYFUSS    ALAN                  27  
  DAY-LEWIS   FRANCES               26  
  DENCH       CHARLIZE              24  
  DEPP        SPENCER               24  
  DEE         LUCILLE               24  
  DAVIS       JENNIFER     

### Text-to-Cipher
In this section we illustrate how to build a Neo4j powered GraphDB and then query it using natural language. The query construction process will convert natural language to a format compatible with Neo4j.

First we'll install a **local instance of Neo4j desktop** [[download link]()], which includes the Neo4j database server, allowing you to run, manage, and connect to local database instances on your machine without relying on external network access

** TODO**

### Text-to-embedding

